In [1]:
%pip install mat73 h5py


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import mat73                      # MATLAB v7.3 (HDF5) 형식 로드용
import scipy.io as sio            # MATLAB v7.2 이하 형식 로드용 (fallback)
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import os
from importlib.util import find_spec
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

In [3]:
# 데이터 경로 설정
DATA_DIR = os.path.expanduser(
    '/Users/eom-jinyong/Downloads/데이터분석_MiniProject/Data'
)

files = sorted(os.listdir(DATA_DIR))
print(f"데이터 경로 : {DATA_DIR}")
print(f"파일 목록 :")
for f in files:
    size_mb = os.path.getsize(os.path.join(DATA_DIR, f)) / (1024**3)
    print(f"   {f}  ({size_mb:.1f} GB)")

데이터 경로 : /Users/eom-jinyong/Downloads/데이터분석_MiniProject/Data
파일 목록 :
   2017-05-12_batchdata_updated_struct_errorcorrect.mat  (2.8 GB)
   2018-02-20_batchdata_updated_struct_errorcorrect.mat  (1.9 GB)
   2018-04-03_varcharge_batchdata_updated_struct_errorcorrect.mat  (0.1 GB)
   2018-04-12_batchdata_updated_struct_errorcorrect.mat  (3.0 GB)


In [4]:
# Batch 1 로드 (가장 많이 사용되는 기본 배치)
batch1_path = os.path.join(DATA_DIR, '2017-05-12_batchdata_updated_struct_errorcorrect.mat')

def load_mat(path):
    """
    .mat 파일 로더 (버전 자동 감지)
    - MATLAB v7.3 (HDF5, 대용량) : mat73.loadmat() 사용
    - MATLAB v7.2 이하          : scipy.io.loadmat() 사용
    """
    try:
        data = mat73.loadmat(path)
        print("로드 완료 (MATLAB v7.3 / HDF5 형식)")
    except Exception:
        data = sio.loadmat(path, simplify_cells=True)
        print("로드 완료 (MATLAB v7.2 이하 형식)")
    return data

print("로딩 중... (파일이 크므로 수십 초 소요될 수 있습니다)")
mat = load_mat(batch1_path)

# 최상위 키 확인
print(f"\n최상위 키 : {list(mat.keys())}")

ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:ro

로딩 중... (파일이 크므로 수십 초 소요될 수 있습니다)
로드 완료 (MATLAB v7.3 / HDF5 형식)

최상위 키 : ['batch', 'batch_date']


In [5]:
batch = mat['batch']

# mat73은 list of dict, scipy는 structured array로 반환
# 아래 코드로 둘 다 동일하게 처리
if isinstance(batch, dict):
    # mat73: {'field': [val0, val1, ...]} 형태 → list of dict로 변환
    keys = list(batch.keys())
    n_cells = len(batch[keys[0]])
    batch = [{k: batch[k][i] for k in keys} for i in range(n_cells)]
    print("mat73 형식 → list of dict 변환 완료")

print(f"배터리 셀 수    : {len(batch)}")
print(f"타입            : {type(batch)}")
print(f"첫 번째 셀 키   : {list(batch[0].keys())}")

mat73 형식 → list of dict 변환 완료
배터리 셀 수    : 46
타입            : <class 'list'>
첫 번째 셀 키   : ['Vdlin', 'barcode', 'channel_id', 'cycle_life', 'cycles', 'policy', 'policy_readable', 'summary']


In [6]:
# 단일 셀 구조 상세 확인
cell0 = batch[0]

print("=" * 55)
print("배터리 셀[0] 구조")
print("=" * 55)

for key, val in cell0.items():
    if isinstance(val, np.ndarray):
        print(f"  {key:25s} | ndarray  | shape={val.shape}")
    elif isinstance(val, dict):
        print(f"  {key:25s} | dict     | keys={list(val.keys())}")
    else:
        print(f"  {key:25s} | {type(val).__name__:8s} | {val}")

배터리 셀[0] 구조
  Vdlin                     | ndarray  | shape=(1000,)
  barcode                   | NoneType | None
  channel_id                | NoneType | None
  cycle_life                | ndarray  | shape=()
  cycles                    | dict     | keys=['I', 'Qc', 'Qd', 'Qdlin', 'T', 'Tdlin', 'V', 'discharge_dQdV', 't']
  policy                    | str      | 3_6C-80PER_3_6C
  policy_readable           | str      | 3.6C(80%)-3.6C
  summary                   | dict     | keys=['IR', 'QCharge', 'QDischarge', 'Tavg', 'Tmax', 'Tmin', 'chargetime', 'cycle']


In [7]:
# cycles 내부 구조 확인 (핵심 시계열 데이터)
cycles_raw = cell0['cycles']

# mat73: cycles도 dict of lists 형태로 반환 → list of dict로 변환
if isinstance(cycles_raw, dict):
    keys = list(cycles_raw.keys())
    n_cycles = len(cycles_raw[keys[0]])
    cycles = [{k: cycles_raw[k][i] for k in keys} for i in range(n_cycles)]
    print(f"mat73 형식 → list of dict 변환 완료")
else:
    cycles = cycles_raw

print(f"총 사이클 수 : {len(cycles)}")
print(f"\n사이클[0] 내 변수 목록 :")
for key, val in cycles[0].items():
    if isinstance(val, np.ndarray):
        print(f"  {key:15s} | ndarray | shape={val.shape} | dtype={val.dtype}")
    else:
        print(f"  {key:15s} | {type(val).__name__:8s} | {val}")

mat73 형식 → list of dict 변환 완료
총 사이클 수 : 1189

사이클[0] 내 변수 목록 :
  I               | NoneType | None
  Qc              | NoneType | None
  Qd              | NoneType | None
  Qdlin           | NoneType | None
  T               | NoneType | None
  Tdlin           | NoneType | None
  V               | NoneType | None
  discharge_dQdV  | NoneType | None
  t               | NoneType | None


In [8]:
def to_list_of_dicts(d):
    """mat73의 dict-of-lists 구조를 list-of-dicts로 변환"""
    if isinstance(d, dict):
        keys = list(d.keys())
        n = len(d[keys[0]])
        return [{k: d[k][i] for k in keys} for i in range(n)]
    return d  # 이미 list 형태면 그대로 반환

def extract_summary(batch):
    """모든 배터리 셀의 summary 데이터를 DataFrame으로 변환"""
    records = []
    for i, cell in enumerate(batch):
        summary = cell['summary']
        # mat73은 summary도 dict-of-lists → 각 필드를 직접 접근
        if isinstance(summary, dict):
            qd   = np.array(summary['QDischarge'])
            qc   = np.array(summary['QCharge'])
            ir   = np.array(summary['IR'])
            tmax = np.array(summary['Tmax'])
            tavg = np.array(summary['Tavg'])
            tmin = np.array(summary['Tmin'])
            ct   = np.array(summary['chargetime'])
        else:
            qd   = summary['QDischarge']
            qc   = summary['QCharge']
            ir   = summary['IR']
            tmax = summary['Tmax']
            tavg = summary['Tavg']
            tmin = summary['Tmin']
            ct   = summary['chargetime']

        raw_life = cell.get("cycle_life")
        life_array = np.asarray(raw_life, dtype=float).reshape(-1)

        cycle_life = (
            float(life_array[0])
            if life_array.size == 1 and np.isfinite(life_array[0])
            else np.nan
            )
        policy     = str(cell.get('policy_readable') or cell.get('policy') or 'unknown')
        n          = len(qd)

        for c in range(n):
            records.append({
                'cell_id'         : i,
                'cycle'           : c + 1,
                'cycle_life'      : cycle_life,
                'charging_policy' : policy,
                'QD'              : qd[c],
                'QC'              : qc[c],
                'IR'              : ir[c],
                'Tmax'            : tmax[c],
                'Tavg'            : tavg[c],
                'Tmin'            : tmin[c],
                'chargetime'      : ct[c],
            })
    return pd.DataFrame(records)

In [9]:
df = extract_summary(batch)

print(f"DataFrame shape : {df.shape}")
df.head()

DataFrame shape : (38811, 11)


,cell_id,cycle,cycle_life,charging_policy,QD,QC,IR,Tmax,Tavg,Tmin,chargetime
0,0,1,1190.0,3.6C(80%)-3.6C,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0,2,1190.0,3.6C(80%)-3.6C,1.070689,1.071042,0.016742,35.652016,31.875011,29.566130,13.341250
2,0,3,1190.0,3.6C(80%)-3.6C,1.071900,1.071674,0.016724,35.692978,31.931490,29.604385,13.425777
3,0,4,1190.0,3.6C(80%)-3.6C,1.072510,1.072304,0.016681,35.680588,31.932603,29.744202,13.425167
4,0,5,1190.0,3.6C(80%)-3.6C,1.073174,1.072970,0.016662,35.728691,31.959322,29.644709,13.341442


In [10]:
# 기본 통계
df.describe().round(3)

,cell_id,cycle,cycle_life,QD,QC,IR,Tmax,Tavg,Tmin,chargetime
count,38811.000,38811.000,38811.000,38811.000,38811.000,38811.000,38811.000,38811.000,38811.000,38811.000
mean,20.771,442.121,884.241,1.045,1.045,0.017,37.322,33.404,30.589,11.510
std,13.469,276.876,186.940,0.059,0.061,0.001,2.760,1.964,1.575,4.518
min,0.000,1.000,534.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
25%,9.000,211.000,731.000,1.036,1.036,0.017,35.961,32.438,29.938,10.653
50%,21.000,422.000,870.000,1.063,1.062,0.017,37.599,33.471,30.390,11.208
75%,32.000,640.000,1017.000,1.076,1.076,0.017,39.107,34.472,31.200,12.092
max,45.000,1226.000,1227.000,2.884,2.966,0.022,43.419,38.414,34.851,419.873


In [11]:
print("결측치 현황 : ")
print(df.isnull().sum())

결측치 현황 : 
cell_id            0
cycle              0
cycle_life         0
charging_policy    0
QD                 0
QC                 0
IR                 0
Tmax               0
Tavg               0
Tmin               0
chargetime         0
dtype: int64


In [12]:
print(f"\n배터리 셀 수   : {df['cell_id'].nunique()}")
print(f"충전 정책 종류 : {df['charging_policy'].nunique()}")


배터리 셀 수   : 46
충전 정책 종류 : 23


# Data PreProcessing

In [14]:
import numpy as np
import matplotlib.pyplot as plt

def calculate_delta_q(cell, cycle_start=10, cycle_end=100):
    cycles = to_list_of_dicts(cell["cycles"])

    # 배열 인덱스와 실제 사이클 번호를 구분
    cycle_numbers = np.asarray(
        cell["summary"]["cycle"], dtype=float
    ).reshape(-1)

    if len(cycle_numbers) != len(cycles):
        raise ValueError(
            "summary의 사이클 수와 cycles 길이가 다릅니다. "
            "사이클 번호 매핑을 확인하세요."
        )

    def get_qdlin(cycle_number):
        matches = np.flatnonzero(cycle_numbers == cycle_number)

        if len(matches) != 1:
            raise ValueError(
                f"사이클 {cycle_number}를 하나로 식별할 수 없습니다."
            )

        # cycles[n]['Qdlin']에서 해당 사이클의 곡선 추출
        n = int(matches[0])
        raw = cycles[n]["Qdlin"]

        if raw is None:
            raise ValueError(f"사이클 {cycle_number}의 Qdlin이 없습니다.")

        return np.asarray(raw, dtype=float).reshape(-1)

    voltage = np.asarray(cell["Vdlin"], dtype=float).reshape(-1)
    q_start = get_qdlin(cycle_start)
    q_end = get_qdlin(cycle_end)

    if not (len(voltage) == len(q_start) == len(q_end)):
        raise ValueError("Vdlin과 두 Qdlin의 길이가 다릅니다.")

    valid = (
        np.isfinite(voltage)
        & np.isfinite(q_start)
        & np.isfinite(q_end)
    )

    if valid.sum() < 2:
        raise ValueError("비교할 수 있는 유효 전압점이 부족합니다.")

    voltage = voltage[valid]
    q_start = q_start[valid]
    q_end = q_end[valid]

    # 전압을 오름차순으로 정렬
    order = np.argsort(voltage)
    voltage = voltage[order]
    q_start = q_start[order]
    q_end = q_end[order]

    delta_q = q_end - q_start

    return voltage, q_start, q_end, delta_q


In [15]:
import numpy as np
import pandas as pd

curves = []
skipped = []

for cell_id, cell in enumerate(batch):
    try:
        life = float(np.asarray(cell["cycle_life"]).item())
        if not np.isfinite(life):
            raise ValueError("유효한 cycle_life가 없습니다.")

        voltage, _, _, delta_q = calculate_delta_q(cell)

        # 보간에 필요한 전압축 확인
        if np.any(np.diff(voltage) <= 0):
            raise ValueError("전압축에 중복값이 있습니다.")

        curves.append((cell_id, life, voltage, delta_q))

    except (ValueError, KeyError, TypeError) as error:
        skipped.append({"cell_id": cell_id, "reason": str(error)})

if not curves:
    raise ValueError("특징을 추출할 수 있는 셀이 없습니다.")

# 모든 셀의 공통 전압 범위
v_min = max(v.min() for _, _, v, _ in curves)
v_max = min(v.max() for _, _, v, _ in curves)

if v_min >= v_max:
    raise ValueError("셀 간 공통 전압 구간이 없습니다.")

voltage_grid = np.linspace(v_min, v_max, 1000)
integrate = np.trapezoid if hasattr(np, "trapezoid") else np.trapz

records = []

for cell_id, life, voltage, delta_q in curves:
    dq = np.interp(voltage_grid, voltage, delta_q)

    group = (
        "Short" if life < 500
        else "Long" if life > 1000
        else "Middle"
    )

    records.append({
        "cell_id": cell_id,
        "cycle_life": life,
        "life_group": group,
        "dq_min": dq.min(),                         # Ah
        "dq_mean": dq.mean(),                       # Ah
        "dq_var": dq.var(),                         # Ah²
        "dq_abs_area": integrate(
            np.abs(dq), voltage_grid
        ),                                         # Ah·V
        "dq_min_voltage": voltage_grid[dq.argmin()], # V
    })

dq_features = pd.DataFrame(records)
display(dq_features.round(6))

feature_cols = [
    "dq_min", "dq_mean", "dq_var",
    "dq_abs_area", "dq_min_voltage",
]

# 그룹별 표본 수와 특징 중앙값
display(dq_features.groupby("life_group").size().rename("n_cells"))
display(
    dq_features.groupby("life_group")[feature_cols]
    .median()
    .round(6)
)

# 연속적인 수명과의 상관관계
correlations = pd.DataFrame({
    "Pearson": dq_features[feature_cols].corrwith(
        dq_features["cycle_life"], method="pearson"
    ),
    "Spearman": dq_features[feature_cols].corrwith(
        dq_features["cycle_life"], method="spearman"
    ),
})

display(
    correlations.sort_values(
        "Spearman", key=lambda x: x.abs(), ascending=False
    ).round(3)
)

if skipped:
    display(pd.DataFrame(skipped))

,cell_id,cycle_life,life_group,dq_min,dq_mean,dq_var,dq_abs_area,dq_min_voltage
0,0,1190.0,Long,-0.008460,-0.002873,0.000010,0.004437,3.064565
1,1,1179.0,Long,-0.011004,-0.004100,0.000010,0.006155,3.136637
2,2,1177.0,Long,-0.017216,-0.004487,0.000018,0.006737,3.153153
3,3,1226.0,Long,-0.018961,-0.007456,0.000036,0.011212,2.957958
4,4,1227.0,Long,-0.013958,-0.005750,0.000023,0.008635,2.878378
5,5,1074.0,Long,-0.025179,-0.011074,0.000066,0.016655,2.872372
6,6,636.0,Middle,-0.037886,-0.015965,0.000170,0.023977,2.933934
7,7,870.0,Middle,-0.038233,-0.016646,0.000154,0.024988,3.060060
8,8,879.0,Middle,-0.030808,-0.012819,0.000100,0.019245,2.941441
9,9,1054.0,Long,-0.028724,-0.011511,0.000087,0.017281,2.968468


life_group
Long      10
Middle    36
Name: n_cells, dtype: int64

,dq_min,dq_mean,dq_var,dq_abs_area,dq_min_voltage
life_group,,,,,
Long,-0.022070,-0.009265,0.000051,0.013933,2.954204
Middle,-0.038294,-0.017209,0.000164,0.025834,2.934685


,Pearson,Spearman
dq_var,-0.838,-0.871
dq_min,0.882,0.850
dq_mean,0.854,0.828
dq_abs_area,-0.854,-0.828
dq_min_voltage,0.312,0.056


In [16]:
# 기존 특징 테이블 복사
df = dq_features.copy()

# dq_min은 기존 컬럼 사용
# 분산이 음수·무한대이면 NaN 처리
valid_var = df['dq_var'].where(
    np.isfinite(df['dq_var']) & (df['dq_var'] >= 0)
)

df['log_dq_var'] = np.log10(valid_var + 1e-12)

In [18]:
df

,cell_id,cycle_life,life_group,dq_min,dq_mean,dq_var,dq_abs_area,dq_min_voltage,log_dq_var
0,0,1190.0,Long,-0.008460,-0.002873,0.000010,0.004437,3.064565,-5.014861
1,1,1179.0,Long,-0.011004,-0.004100,0.000010,0.006155,3.136637,-5.013960
2,2,1177.0,Long,-0.017216,-0.004487,0.000018,0.006737,3.153153,-4.737000
3,3,1226.0,Long,-0.018961,-0.007456,0.000036,0.011212,2.957958,-4.442613
4,4,1227.0,Long,-0.013958,-0.005750,0.000023,0.008635,2.878378,-4.647744
5,5,1074.0,Long,-0.025179,-0.011074,0.000066,0.016655,2.872372,-4.178878
6,6,636.0,Middle,-0.037886,-0.015965,0.000170,0.023977,2.933934,-3.768878
7,7,870.0,Middle,-0.038233,-0.016646,0.000154,0.024988,3.060060,-3.813486
8,8,879.0,Middle,-0.030808,-0.012819,0.000100,0.019245,2.941441,-4.001195
9,9,1054.0,Long,-0.028724,-0.011511,0.000087,0.017281,2.968468,-4.059384
